In [11]:
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"
import pandas as pd
import numpy as np
from pathlib import Path

In [12]:
#=========
# Setup
#=========
df = pd.DataFrame({
    "date": pd.to_datetime(["2025-11-01" , "2025-11-02" , "2025-11-03" , "2025-11-04"]) ,
    "region": ["East" , "West" , "East" , "North"] ,
    "sales": [120 , 80 , 150 , 90] ,
    "returns": [5 , 2 , 4 , 1] ,
})
df["return_rate"] = (df["returns"] / df["sales"]).round(3)
df

out = Path("exports_day30")
out.mkdir(exist_ok = True)
out

,date,region,sales,returns,return_rate
0,2025-11-01,East,120,5,0.042
1,2025-11-02,West,80,2,0.025
2,2025-11-03,East,150,4,0.027
3,2025-11-04,North,90,1,0.011


PosixPath('exports_day30')

In [13]:
#====================
# Case 1) CSV read
#====================
csv_path = out / "sales_raw.csv"
csv_path
df.assign(sales = df["sales"].astype(str)).to_csv(csv_path , index = False)

df_csv = pd.read_csv(
    csv_path , parse_dates = ["date"] , dtype = {"region": "string"} ,
    na_values = ["" , "NA" , "N/A"] ,
)
df_csv["sales"] = pd.to_numeric(df_csv["sales"] , errors = "coerce")
df_csv
df_csv.dtypes

PosixPath('exports_day30/sales_raw.csv')

,date,region,sales,returns,return_rate
0,2025-11-01,East,120,5,0.042
1,2025-11-02,West,80,2,0.025
2,2025-11-03,East,150,4,0.027
3,2025-11-04,North,90,1,0.011


date           datetime64[ns]
region         string[python]
sales                   int64
returns                 int64
return_rate           float64
dtype: object

In [14]:
#========================================================
# Case 2) Parquet (schema-preserving analytics storage)
#========================================================
parquet_path = out / "sales.parquet"
parquet_path
df.to_parquet(parquet_path , index = False)
df_pq = pd.read_parquet(parquet_path)
df_pq
df_pq.dtypes

PosixPath('exports_day30/sales.parquet')

,date,region,sales,returns,return_rate
0,2025-11-01,East,120,5,0.042
1,2025-11-02,West,80,2,0.025
2,2025-11-03,East,150,4,0.027
3,2025-11-04,North,90,1,0.011


date           datetime64[ns]
region                 object
sales                   int64
returns                 int64
return_rate           float64
dtype: object

In [15]:
#============================================
# Case 3) Excel export (multi-sheet report)
#============================================
xlsx_path = out / "report.xlsx"
xlsx_path
summary = (
    df.groupby("region" , as_index = False)
    .agg(total_sales = ("sales" , "sum") , avg_return_rate = ("return_rate" , "mean"))
)
with pd.ExcelWriter(xlsx_path , engine = "openpyxl") as writer:
    summary.to_excel(writer , sheet_name = "Summary" , index = False)
    df.to_excel(writer , sheet_name = "Detail" , index = False)
summary

PosixPath('exports_day30/report.xlsx')

,region,total_sales,avg_return_rate
0,East,270,0.0345
1,North,90,0.0110
2,West,80,0.0250


In [16]:
#=============================
# Case 4) JSON for APIs/apps
#=============================
json_path = out / "latest_metrics.json"
json_path
payload = df.sort_values("date").tail(2)
payload.to_json(json_path , orient = "records" , date_format = "iso")
payload
df_json = pd.read_json(json_path)
df_json.head(2)

PosixPath('exports_day30/latest_metrics.json')

,date,region,sales,returns,return_rate
2,2025-11-03,East,150,4,0.027
3,2025-11-04,North,90,1,0.011


,date,region,sales,returns,return_rate
0,2025-11-03,East,150,4,0.027
1,2025-11-04,North,90,1,0.011


In [17]:
#=======================================
# Case 5) SQL handoff (SQLite example)
#=======================================
import sqlite3
db_path = out / "metrics.db"
db_path
con = sqlite3.connect(db_path)
con

df.to_sql("sales_metrics" , con , if_exists = "replace" , index = False)
df

df_sql = pd.read_sql(
    "SELECT region, SUM(sales) AS total_sales FROM sales_metrics GROUP BY region",
    con
)
df_sql
con.close()

PosixPath('exports_day30/metrics.db')

4

,date,region,sales,returns,return_rate
0,2025-11-01,East,120,5,0.042
1,2025-11-02,West,80,2,0.025
2,2025-11-03,East,150,4,0.027
3,2025-11-04,North,90,1,0.011


,region,total_sales
0,East,270
1,North,90
2,West,80


In [18]:
#================================
# Case 6) Clean export pattern
#================================
run_tag = pd.Timestamp.today().strftime("%Y%m%d")
export_path = out / f"kpi_export_{run_tag}.csv"
export_path

cols = ["date" , "region" , "sales" , "returns" , "return_rate"]
df[cols].to_csv(export_path , index = False)
df

PosixPath('exports_day30/kpi_export_20260116.csv')

,date,region,sales,returns,return_rate
0,2025-11-01,East,120,5,0.042
1,2025-11-02,West,80,2,0.025
2,2025-11-03,East,150,4,0.027
3,2025-11-04,North,90,1,0.011
